# EC1 &middot; Anonimização de *tickets* de incidentes

**Minicurso "Inteligência Artificial aplicada à Resposta a Incidentes" &middot; SBSeg 2026**

Este *notebook* acompanha a Seção 1.5 do capítulo. Ele reimplementa, em cerca de
cem linhas de Python puro, o mecanismo essencial do **AnonShield**: detecção
híbrida de entidades em texto livre e pseudonimização determinística com
HMAC-SHA256, com base de mapeamento local e reidentificação auditada.

Não substitui a ferramenta de produção
([AnonShield](https://github.com/AnonShield/tool)), que trata dezenas de
formatos, opera em fluxo contínuo e processa dezenas de milhares de registros
por minuto. O propósito aqui é outro: **tornar inspecionável** o que a
ferramenta faz.

## O que você vai responder

| Etapa | Pergunta |
|---|---|
| 1 | Que entidades um analista humano marcaria como sensíveis neste *ticket*? |
| 2 | O que o detector vê que eu não vi, e o que eu vi que ele não viu? |
| 3 | **Quais entidades _apenas_ o NER captura?** (a etapa mais importante) |
| 4 | O pseudônimo preserva o tipo, e o tipo preserva a classificabilidade? |
| 5 | A mesma entidade recebe o mesmo pseudônimo em *tickets* diferentes? |
| 6 | Como reverter, e o que a governança exige em troca? |

**Saída:** `saida/tickets_pseudonimizados.csv`, que é a entrada do *notebook* 02.

> **Nenhum dado real.** Os *tickets* de `dados/tickets_sinteticos.csv` são
> fictícios; os IPs vêm das faixas de documentação (RFC 5737).

## Configuração

In [ ]:
import csv, hashlib, hmac, ipaddress, os, re, sqlite3, unicodedata
from datetime import datetime, timezone
from pathlib import Path

# Raiz do repositório (funciona tanto de notebooks/ quanto da raiz)
RAIZ = Path.cwd()
if not (RAIZ / "dados").is_dir():
    RAIZ = RAIZ.parent
DADOS = RAIZ / "dados"
SAIDA = RAIZ / "saida"
SAIDA.mkdir(exist_ok=True)

# --- Chave HMAC do operador ----------------------------------------------
# Em produção: variável de ambiente ou cofre de segredos. NUNCA versionada,
# NUNCA compartilhada junto com a base pseudonimizada. Aqui usamos uma chave
# fixa para que o notebook seja reprodutível em sala.
CHAVE = os.environ.get("MC1_HMAC_KEY", "chave-de-sala-de-aula-nao-use-em-producao").encode()
TAMANHO_PSEUDONIMO = 10          # caracteres hexadecimais do slug

# --- NER por transformer (opcional) --------------------------------------
# True exige `pip install transformers torch` e baixa ~1 GB na primeira
# execução. Com False, usamos um reconhecedor léxico simplificado: a lição
# da etapa 3 (o que só o NER captura) continua observável.
USAR_TRANSFORMER = False
MODELO_NER = "Davlan/xlm-roberta-base-ner-hrl"

print("raiz:", RAIZ)
print("chave HMAC carregada:", "sim" if CHAVE else "nao",
      f"({len(CHAVE)} bytes)")
print("NER por transformer:", USAR_TRANSFORMER)

## Etapa 1 &middot; Leitura e inspeção

Antes de rodar qualquer detector, **leia um *ticket* como analista**. Anote
mentalmente o que você tiraria antes de mandar esse texto para um modelo de
linguagem de terceiro. Esse é o gabarito informal contra o qual o código será
avaliado nas próximas etapas.

In [ ]:
with open(DADOS / "tickets_sinteticos.csv", encoding="utf-8") as f:
    TICKETS = list(csv.DictReader(f))

print(f"{len(TICKETS)} tickets carregados")
print("categorias presentes:",
      ", ".join(sorted({t["categoria"] for t in TICKETS},
                       key=lambda c: int(c[3:]))))
print("=" * 72)
print(TICKETS[0]["texto"])

## Etapa 2 &middot; Detecção de entidades

O problema de detecção em texto livre divide-se em dois regimes, e essa divisão
é a razão de ser da abordagem híbrida:

- **Formato fixo** (IP, e-mail, URL, CVE, *hash*, telefone): expressões
  regulares resolvem, com precisão alta e custo desprezível.
- **Sem padrão sintático** (nomes de pessoas e de organizações): exige
  reconhecimento de entidades nomeadas.

Repare no que **não** é detectado de propósito: número de sistema autônomo,
porta, protocolo, severidade CVSS e datas. São exatamente os elementos que
sustentam a classificação do EC2. Anonimizar não é apagar tudo: é apagar o que
identifica e preservar o que classifica.

In [ ]:
# --- Regime 1: entidades de formato fixo ---------------------------------
# A ordem importa: EMAIL antes de URL e de HOSTNAME, senão o domínio do
# e-mail seria capturado isoladamente e o pseudônimo perderia coerência.
# Cada entrada é (tipo, expressão, validador). A expressão apenas *localiza*
# o candidato; quem decide se ele é válido é o validador. Essa separação
# elimina uma classe inteira de falsos positivos: sem ela, a expressão de
# IPv6 casaria com o carimbo de hora "03:14:22" de um log de SSH.
def _e_ipv4(s):
    try:
        return ipaddress.ip_address(s).version == 4
    except ValueError:
        return False


def _e_ipv6(s):
    try:
        return ipaddress.ip_address(s).version == 6
    except ValueError:
        return False


PADROES = [
    ("EMAIL_ADDRESS", re.compile(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b"), None),
    ("URL", re.compile(r"https?://[^\s<>\"')\]]+"), None),
    ("HASH", re.compile(
        r"\b[a-fA-F0-9]{64}\b|\b[a-fA-F0-9]{40}\b|\b[a-fA-F0-9]{32}\b"), None),
    ("IP_ADDRESS", re.compile(r"\b\d{1,3}(?:\.\d{1,3}){3}\b"), _e_ipv4),
    ("IPV6_ADDRESS", re.compile(
        r"\b[0-9A-Fa-f]{0,4}(?::[0-9A-Fa-f]{0,4}){2,7}\b"), _e_ipv6),
    ("PHONE", re.compile(r"\+55\s?\d{2}\s?\d{4,5}-?\d{4}"), None),
    ("HOSTNAME", re.compile(
        r"\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+"
        r"(?:br|com|org|net|edu|gov|info|io|dev|example|exemplo)\b"), None),
    ("WINDOWS_USER_PATH", re.compile(r"[A-Z]:\\Users\\[A-Za-z0-9._-]+"), None),
]

# Deliberadamente NÃO detectados: preservam o sinal analítico do relato.
PRESERVADOS = ["ASN", "porta/protocolo", "CVE", "CPE", "CVSS", "datas",
               "vocabulário técnico"]


def detectar_regex(texto):
    """Devolve [(inicio, fim, tipo, valor)] sem sobreposições."""
    achados = []
    for tipo, padrao, validador in PADROES:
        for m in padrao.finditer(texto):
            if validador is None or validador(m.group(0)):
                achados.append((m.start(), m.end(), tipo, m.group(0)))
    return _resolver_sobreposicoes(achados)


def _resolver_sobreposicoes(achados):
    """Mantém o primeiro achado de cada região; empates vão para o mais longo.

    A ordem de PADROES define a prioridade, e por isso a lista chega aqui
    já ordenada por tipo. Ordenamos por (inicio, -comprimento) e descartamos
    tudo que colidir com uma região já aceita.
    """
    aceitos, ocupadas = [], []
    for ini, fim, tipo, valor in sorted(achados, key=lambda a: (a[0], -(a[1] - a[0]))):
        if any(ini < o_fim and fim > o_ini for o_ini, o_fim in ocupadas):
            continue
        aceitos.append((ini, fim, tipo, valor))
        ocupadas.append((ini, fim))
    return sorted(aceitos)

In [ ]:
# --- Regime 2: entidades sem padrão sintático (NER) ----------------------
# Dois caminhos. O primeiro é um modelo transformer multilíngue, que é o que
# a ferramenta de produção usa. O segundo é um reconhecedor léxico
# simplificado, escrito para este notebook, que funciona sem dependências.
#
# O reconhecedor léxico é assumidamente pior: erra em nomes fora de padrão e
# em frases que começam com maiúscula. Ele existe para que a etapa 3 seja
# executável em qualquer máquina, não como proposta de método.

# Conectivos que ligam partes de um nome próprio ("Marta Lopes de Souza").
# Deliberadamente curto: incluir "por" faria "Registrado por Tatiana Moraes"
# virar um único nome.
_CONECTIVOS = {"de", "da", "do", "das", "dos", "e"}
# Termos técnicos que começam com maiúscula e NÃO são nome de pessoa ou de
# organização. Sem esta lista, o reconhecedor léxico produziria falsos
# positivos em profusão.
_TECNICO = {
    "Assunto","Prezados","Para","De","Data","Origem","Destino","Alvo","Vetor",
    "Pico","Contato","Hash","URL","IP","ASN","CVE","CPE","CVSS","TCP","UDP",
    "SSH","DNS","NTP","RDP","HTTP","HTTPS","SQL","GPU","CPU","RAM","EDR","IDS",
    "SIEM","SOAR","WAF","CMS","API","PIX","CPF","Log","Logs","Feb","Failed",
    "Accepted","Recomendacao","Recomendamos","Mitigacao","Acoes","Nao","Nenhuma",
    "Evidencia","Responsavel","Vulnerabilidade","Ticket","Subject","Hello",
    "Please","Server","The","Critical","Internet","Windows","Apache","WordPress",
    "Ansible","Markdown","Slurm","BGP","ANPD","LGPD","Boa","Notamos","Duracao",
    "Periodo","Arquivo","Commit","Servicos","Aberto","Solicitamos","Segundo",
    "Foram","Durante","Nosso","Nossa","Entre","Entrada","Primeiro","Analista",
    "Notificacao","Notificação","Registrado","Remote","Code","Execution",
    "UNION","SELECT","Vulnerabilidade","Mitigado","Alerta","Alvo",
}
_MARCADOR_ORG = {"Universidade","Instituto","Fundacao","Fundação","Empresa",
                 "Colegio","Colégio","Prefeitura","Hospital","Ltda","Reitoria",
                 "Datacenter","Comissao","Comissão"}

_TOKEN_MAIUSCULO = re.compile(r"\b[A-ZÀ-Ý][A-Za-zÀ-ÿ'-]+\b")


def _ner_lexico(texto):
    """Reconhecedor de PERSON e ORGANIZATION baseado em capitalização."""
    achados, i = [], 0
    tokens = list(_TOKEN_MAIUSCULO.finditer(texto))
    while i < len(tokens):
        if tokens[i].group(0) in _TECNICO:
            i += 1
            continue
        grupo, j = [tokens[i]], i + 1
        # Estende enquanto os tokens forem adjacentes (só espaço ou conectivo)
        while j < len(tokens):
            entre = texto[tokens[j - 1].end():tokens[j].start()]
            if entre == " ":
                if tokens[j].group(0) in _TECNICO:
                    break
                grupo.append(tokens[j]); j += 1
            elif re.fullmatch(r" (?:de|da|do|das|dos|e) ", entre):
                grupo.append(tokens[j]); j += 1
            else:
                break
        if len(grupo) >= 2:
            ini, fim = grupo[0].start(), grupo[-1].end()
            trecho = texto[ini:fim]
            tipo = ("ORGANIZATION"
                    if any(t.group(0) in _MARCADOR_ORG for t in grupo)
                    else "PERSON")
            achados.append((ini, fim, tipo, trecho))
            i = j
        else:
            i += 1
    return achados


_pipeline_ner = None


def _ner_transformer(texto):
    global _pipeline_ner
    if _pipeline_ner is None:
        from transformers import pipeline
        _pipeline_ner = pipeline("token-classification", model=MODELO_NER,
                                 aggregation_strategy="simple")
    mapa = {"PER": "PERSON", "ORG": "ORGANIZATION", "LOC": "LOCATION"}
    saida = []
    for e in _pipeline_ner(texto):
        tipo = mapa.get(e["entity_group"])
        if tipo and e["score"] >= 0.80:
            saida.append((e["start"], e["end"], tipo,
                          texto[e["start"]:e["end"]]))
    return saida


def detectar_ner(texto):
    if USAR_TRANSFORMER:
        try:
            return _ner_transformer(texto)
        except Exception as erro:                      # noqa: BLE001
            print(f"[aviso] transformer indisponivel ({erro}); "
                  "usando reconhecedor lexico")
    return _ner_lexico(texto)


def detectar(texto, com_ner=True):
    """Detecção híbrida. com_ner=False é a ablação da Etapa 3."""
    achados = detectar_regex(texto)
    if com_ner:
        ocupadas = [(i, f) for i, f, _, _ in achados]
        for ini, fim, tipo, valor in detectar_ner(texto):
            if not any(ini < of and fim > oi for oi, of in ocupadas):
                achados.append((ini, fim, tipo, valor))
                ocupadas.append((ini, fim))
    return sorted(achados)

In [ ]:
def tabela(linhas, cabecalho):
    """Impressão tabular sem depender de pandas."""
    try:
        import pandas as pd
        return pd.DataFrame(linhas, columns=cabecalho)
    except ImportError:
        larg = [max(len(str(c)), *(len(str(l[i])) for l in linhas or [cabecalho]))
                for i, c in enumerate(cabecalho)]
        sep = "  ".join("-" * w for w in larg)
        print("  ".join(str(c).ljust(w) for c, w in zip(cabecalho, larg)))
        print(sep)
        for l in linhas:
            print("  ".join(str(v).ljust(w) for v, w in zip(l, larg)))
        return None


TICKET = TICKETS[0]["texto"]
achados = detectar(TICKET)

print(f"{len(achados)} entidades detectadas\n")
tabela([(t, repr(v)[:46], i, f) for i, f, t, v in achados],
       ["tipo", "valor", "inicio", "fim"])

### Pergunta da etapa 2

Compare a lista acima com o que você anotou na etapa 1.

- **O que o detector viu e você não viu?** Tipicamente o telefone e o segundo
  e-mail, escondidos no rodapé.
- **O que você viu e ele não viu?** É onde estão os limites reais da abordagem.

## Etapa 3 &middot; Ablação do NER

Esta é a etapa pedagogicamente mais importante do *notebook*. Vamos rodar a
detecção **sem** o componente de reconhecimento de entidades nomeadas e ver o
que sobra exposto.

In [ ]:
com_ner  = detectar(TICKET, com_ner=True)
sem_ner  = detectar(TICKET, com_ner=False)

tipos_com = {t for _, _, t, _ in com_ner}
tipos_sem = {t for _, _, t, _ in sem_ner}
so_ner    = sorted(tipos_com - tipos_sem)

print(f"entidades com NER ..: {len(com_ner)}   tipos: {len(tipos_com)}")
print(f"entidades sem NER ..: {len(sem_ner)}   tipos: {len(tipos_sem)}")
print(f"\nCapturado APENAS pelo NER: {so_ner}\n")

perdidas = [(t, v) for i, f, t, v in com_ner
            if t in so_ner]
tabela(perdidas, ["tipo perdido sem NER", "valor que ficaria exposto"])

Rode a célula acima sobre alguns *tickets* diferentes (troque `TICKETS[0]` por
`TICKETS[3]`, `TICKETS[8]`, `TICKETS[15]`) e observe o padrão: o que escapa da
substituição puramente baseada em regras são **nomes de pessoas e de
organizações**, precisamente as entidades cuja exposição causa maior dano.

É por isso que a abordagem híbrida não é conveniência de implementação, e sim
necessidade estrutural. Um tratamento *ad hoc* com `sed` e uma lista de
expressões regulares, comum em CSIRTs, protege o que é fácil e deixa passar o
que importa.

> **Leia o resultado com honestidade.** Com `USAR_TRANSFORMER = False`, quem
> encontra as pessoas e organizações é o reconhecedor léxico deste *notebook*,
> que é bem pior que um modelo de verdade: erra em nomes fora do padrão
> Nome+Sobrenome e produz falsos positivos. Ligue `USAR_TRANSFORMER = True` na
> célula de configuração para ver a diferença. A conclusão qualitativa (só o
> NER pega PERSON e ORGANIZATION) vale nos dois casos; os números, não.

## Etapa 4 &middot; Pseudonimização determinística

Três decisões de projeto, e as consequências de cada uma:

| Decisão | Consequência |
|---|---|
| **HMAC-SHA256 com chave secreta**, em vez de *hash* público | sem a chave, o adversário não reproduz o mapeamento, mesmo conhecendo o algoritmo e o domínio de entrada (crítico para IPv4, cujo espaço é pequeno) |
| **Determinístico**, em vez de aleatório | a mesma entidade recebe o mesmo pseudônimo entre registros, o que preserva a correlação necessária para detectar varredura e movimentação lateral |
| **Tipo preservado no prefixo** | `[IP_ADDRESS c04a9e21]` mantém o relato classificável: o que importa para o EC2 é que existe um IP com DNS recursivo aberto, não qual é o IP |

A terceira decisão é a que articula o EC1 com o EC2. Sem ela, o *pipeline*
inteiro deixa de funcionar.

In [ ]:
def _normalizar(valor):
    """Normalização antes do HMAC: sem ela, 'Joana Alves' e 'joana alves'
    receberiam pseudônimos distintos e a correlação se perderia."""
    v = unicodedata.normalize("NFKD", valor.strip().lower())
    return "".join(c for c in v if not unicodedata.combining(c))


def pseudonimo(tipo, valor, chave=CHAVE, tamanho=TAMANHO_PSEUDONIMO):
    msg = f"{tipo}:{_normalizar(valor)}".encode()
    digesto = hmac.new(chave, msg, hashlib.sha256).hexdigest()
    return f"[{tipo} {digesto[:tamanho]}]", digesto


def anonimizar(texto, com_ner=True, chave=CHAVE, registrar=None):
    """Substitui da direita para a esquerda, para não invalidar os offsets."""
    saida = texto
    for ini, fim, tipo, valor in sorted(detectar(texto, com_ner), reverse=True):
        rotulo, digesto = pseudonimo(tipo, valor, chave)
        saida = saida[:ini] + rotulo + saida[fim:]
        if registrar is not None:
            registrar.append((tipo, valor, rotulo, digesto))
    return saida


print("ANTES\n" + "-" * 72)
print(TICKET)
print("\nDEPOIS\n" + "-" * 72)
print(anonimizar(TICKET))

### Pergunta da etapa 4

O relato continua classificável? Leia a versão pseudonimizada como se fosse a
primeira vez. Você consegue dizer que se trata de um resolvedor DNS recursivo
aberto, passível de uso em amplificação? Se sim, o EC2 também consegue.

## Etapa 5 &middot; Determinismo, correlação e rotação de chave

Duas propriedades opostas e igualmente necessárias:

1. Com a **mesma chave**, a mesma entidade produz o mesmo pseudônimo em
   *tickets* distintos. É isso que permite correlacionar incidentes.
2. Com uma **chave diferente**, todos os pseudônimos mudam. É isso que impede
   que dois parceiros que receberam bases distintas as cruzem, e é também o
   custo da rotação de chave: ela invalida a correlação com todo o material
   pseudonimizado anteriormente.

In [ ]:
t1 = "O host 203.0.113.55 responde a consultas recursivas."
t2 = "Novo evento envolvendo 203.0.113.55 em 12/03, mesma origem."

a1, a2 = anonimizar(t1), anonimizar(t2)
print(a1); print(a2)

extrair = lambda s: re.search(r"\[IP_ADDRESS ([0-9a-f]+)\]", s).group(1)
assert extrair(a1) == extrair(a2), "determinismo quebrado"
print(f"\nOK: mesmo pseudonimo nos dois tickets -> {extrair(a1)}")

# Rotação de chave: a correlação com o material anterior é perdida.
outra = b"chave-rotacionada-em-2026-03"
a1_nova = anonimizar(t1, chave=outra)
print(f"\napos rotacao da chave -> {extrair(a1_nova)}")
print("correlacao com o material anterior:",
      "preservada" if extrair(a1_nova) == extrair(a1) else "PERDIDA (esperado)")

## Etapa 6 &middot; Persistência, reidentificação e trilha de auditoria

Um CSIRT precisa reverter a pseudonimização em três situações: notificar o
responsável por um ativo comprometido, atender a requisição legal fundamentada,
ou correlacionar o registro com evidência bruta preservada.

A reversão exige **dois elementos simultaneamente**: a base de mapeamento *e* a
chave HMAC. A separação é deliberada. Entregar a base pseudonimizada a um
parceiro não lhe confere capacidade de reversão, ainda que ele tenha a
ferramenta e conheça o algoritmo.

E ela cobra três controles de governança:

1. **Controle de acesso**: base e chave em ambiente segregado, nunca
   acompanhando a base pseudonimizada em compartilhamentos.
2. **Trilha de auditoria**: quem, quando, sobre qual entidade e sob qual
   justificativa. Sem isso, a reidentificação vira canal lateral não
   controlado.
3. **Ciclo de vida da chave**: política explícita de rotação, retenção e
   reprocessamento.

In [ ]:
BANCO = SAIDA / "mapeamento.sqlite"
if BANCO.exists():
    BANCO.unlink()
con = sqlite3.connect(BANCO)
con.executescript("""
CREATE TABLE mapeamento (
    tipo TEXT, valor_original TEXT, pseudonimo TEXT, digesto TEXT,
    primeira_ocorrencia TEXT, ultima_ocorrencia TEXT,
    PRIMARY KEY (tipo, valor_original));
CREATE TABLE auditoria (
    momento TEXT, operador TEXT, acao TEXT, alvo TEXT, justificativa TEXT);
""")


def agora():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def registrar_mapeamento(registros):
    for tipo, valor, rotulo, digesto in registros:
        con.execute("""
            INSERT INTO mapeamento VALUES (?,?,?,?,?,?)
            ON CONFLICT(tipo, valor_original)
            DO UPDATE SET ultima_ocorrencia = excluded.ultima_ocorrencia
        """, (tipo, valor, rotulo, digesto, agora(), agora()))
    con.commit()


def reidentificar(pseudonimo_texto, operador, justificativa):
    """Reversão auditada. Sem justificativa, recusa a operação."""
    if not justificativa or len(justificativa) < 10:
        raise ValueError("reidentificacao exige justificativa registrada")
    linha = con.execute(
        "SELECT tipo, valor_original FROM mapeamento WHERE pseudonimo = ?",
        (pseudonimo_texto,)).fetchone()
    con.execute("INSERT INTO auditoria VALUES (?,?,?,?,?)",
                (agora(), operador, "reidentificacao", pseudonimo_texto,
                 justificativa))
    con.commit()
    return linha


registros = []
anonimizar(TICKET, registrar=registros)
registrar_mapeamento(registros)

alvo = con.execute(
    "SELECT pseudonimo FROM mapeamento WHERE tipo='IP_ADDRESS'").fetchone()[0]
print("revertendo:", alvo)
print("resultado  :", reidentificar(
    alvo, operador="analista.n2",
    justificativa="notificacao ao responsavel pelo ativo, caso INC-2026-0001"))

try:
    reidentificar(alvo, "analista.n1", "urgente")
except ValueError as e:
    print("\nrecusado, como esperado:", e)

print("\n--- trilha de auditoria ---")
tabela(con.execute("SELECT momento, operador, alvo, justificativa "
                   "FROM auditoria").fetchall(),
       ["momento", "operador", "alvo", "justificativa"])

> **A consequência legal, sem rodeios.** Como a reversão é possível, o dado
> pseudonimizado **permanece dado pessoal** perante a LGPD. A pseudonimização é
> medida de segurança robusta e boa prática de minimização de risco, mas não
> retira o tratamento do escopo da lei. Equipes que tratam pseudonimização como
> equivalente a anonimização assumem risco de conformidade que não percebem.

## Etapa 7 &middot; Processar a base e gerar a entrada do EC2

In [ ]:
from collections import Counter

todos, contagem = [], Counter()
for t in TICKETS:
    reg = []
    texto_pseudo = anonimizar(t["texto"], registrar=reg)
    registrar_mapeamento(reg)
    contagem.update(tipo for tipo, _, _, _ in reg)
    todos.append({"id": t["id"], "id_incidente": t["id_incidente"],
                  "texto": texto_pseudo, "categoria": t["categoria"]})

destino = SAIDA / "tickets_pseudonimizados.csv"
with open(destino, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["id", "id_incidente", "texto", "categoria"],
                       quoting=csv.QUOTE_ALL)
    w.writeheader(); w.writerows(todos)

distintas = con.execute("SELECT COUNT(*) FROM mapeamento").fetchone()[0]
print(f"{len(todos)} tickets pseudonimizados -> {destino.relative_to(RAIZ)}")
print(f"{sum(contagem.values())} ocorrencias, {distintas} entidades distintas\n")
tabela(sorted(contagem.items(), key=lambda kv: -kv[1]),
       ["tipo de entidade", "ocorrencias"])

## Discussão e limites

**Anonimizar é, antes de tudo, um problema de detecção.** A substituição em si é
trivial: um HMAC-SHA256 resolve o problema criptográfico em uma linha. A
dificuldade está em *localizar*, em texto livre, multilíngue e misturado a
artefatos técnicos, tudo aquilo que precisa ser substituído.

**Três limitações permanecem, e nenhuma some com mais código:**

1. **Cobertura.** O que o NER não reconhece permanece exposto. Na avaliação do
   AnonShield sobre relatórios de vulnerabilidade, os falsos negativos
   residuais concentram-se em nomes de organização em cadeias formulaicas de
   atribuição e em nomes de *host* parciais embutidos em prosa. Rode este
   *notebook* sobre o *ticket* 24 (`TICKETS[23]`), que é vago e mal redigido, e
   veja o que escapa.
2. **Robustez de formato.** PDFs digitalizados dependem inteiramente de OCR.
3. **Reidentificação por inferência.** Mesmo com todos os identificadores
   substituídos, a combinação de janela temporal, tipo de serviço, porte da
   organização e vocabulário do relato pode reduzir drasticamente o espaço de
   candidatos. É o resultado clássico de Sweeney e a demonstração de Narayanan
   e Shmatikov. Por isso os *tickets* pseudonimizados do minicurso são
   compartilhados sob acordo, não publicados.

**Os falsos positivos desta execução são instrutivos.** Consulte a base de
mapeamento e você encontrará, entre os `PERSON`, dois cargos que não são nome
de pessoa: `Encarregada de Dados` e `Pro-Reitor de Administracao`; e um
`ORGANIZATION` truncado por quebra de linha, `Patrimonio da Prefeitura de
Vila`. São limitações do reconhecedor léxico, não do método híbrido, e a
avaliação do AnonShield reporta padrão análogo com modelo de verdade: cadeias
de versão lidas como endereço IP, caminhos de arquivo lidos como nome de
*host*. O denominador comum é a **ambiguidade sintática do próprio texto
técnico**, não o algoritmo de substituição.

```python
import sqlite3
con = sqlite3.connect(SAIDA / "mapeamento.sqlite")
list(con.execute("SELECT tipo, valor_original FROM mapeamento "
                 "WHERE tipo IN ('PERSON','ORGANIZATION')"))
```

**O *recall* é a métrica crítica, não a precisão.** Um falso negativo é uma
entidade sensível que ficou exposta; um falso positivo apenas degrada a
utilidade analítica. Os custos são assimétricos, e a configuração deve refletir
essa assimetria. É por isso que a estratégia `presidio` do AnonShield, com
precisão de 71,9% mas *recall* de 96,4%, é defensável em ambiente de alta
sensibilidade.

## Exercícios

1. Ligue `USAR_TRANSFORMER = True` e compare o conjunto de entidades detectadas
   com o do reconhecedor léxico. Onde o modelo acerta o que a heurística erra?
   E onde ele erra?
2. Acrescente um padrão para CPF (`\d{3}\.\d{3}\.\d{3}-\d{2}`) e verifique se o
   *ticket* 9 passa a ser tratado corretamente. Por que ele não era, antes?
3. Reduza `TAMANHO_PSEUDONIMO` para 3. Procure colisões na base de mapeamento.
   Qual é o compromisso entre legibilidade do pseudônimo e risco de colisão?
4. O *ticket* 4 contém `C:\Users\rcfarias\...`. O padrão `WINDOWS_USER_PATH`
   substitui o caminho inteiro. Isso destrói informação útil para a resposta?
   Como você reescreveria o padrão para preservar a estrutura do caminho e
   pseudonimizar apenas o nome do usuário?

---

**Próximo passo:** abra `02-classificacao.ipynb`, que consome
`saida/tickets_pseudonimizados.csv`.